## LSTM MODEL

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter

all_text = " ".join(df['review'].tolist()).lower().split()
vocab = ["<PAD>", "<UNK>"] + [word for word, count in Counter(all_text).most_common()]
word_to_ix = {word: i for i, word in enumerate(vocab)}

def encode_text(text, max_len=15):
    tokens = text.lower().split()
    encoded = [word_to_ix.get(t, 1) for t in tokens]
    if len(encoded) < max_len:
        encoded += [0] * (max_len - len(encoded))
    else:
        encoded = encoded[:max_len]
    return encoded

class SentimentDataset(Dataset):
    def __init__(self, dataframe, max_len=15):
        self.labels = dataframe['sentiment'].values
        self.texts = [encode_text(t, max_len) for t in dataframe['review']]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.tensor(self.texts[idx]), torch.tensor(self.labels[idx])

lstm_dataset = SentimentDataset(df)
lstm_loader = DataLoader(lstm_dataset, batch_size=2, shuffle=True)

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        out = self.fc(hidden[-1])
        return out

model_lstm = LSTMClassifier(len(vocab), embed_dim=16, hidden_dim=16, output_dim=3)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_lstm.parameters(), lr=0.01)

model_lstm.train()
for epoch in range(5):
    total_loss = 0
    for texts, labels in lstm_loader:
        optimizer.zero_grad()
        outputs = model_lstm(texts)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"LSTM Epoch {epoch+1} Loss: {total_loss/len(lstm_loader):.4f}")

LSTM Epoch 1 Loss: 1.1111
LSTM Epoch 2 Loss: 1.0985
LSTM Epoch 3 Loss: 1.0927
LSTM Epoch 4 Loss: 1.0922
LSTM Epoch 5 Loss: 1.0839


In [ ]:
FINE-TUNING BERT(VIA HUGGINGFACE)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset as HFDataset

hf_dataset = HFDataset.from_pandas(df)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["review"], padding="max_length", truncation=True, max_length=32)

tokenized_dataset = hf_dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.rename_column("sentiment", "label")

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [ ]:
model_bert = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=1,
    report_to="none"
)

trainer = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=tokenized_dataset,
)

trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(load

Step,Training Loss
1,1.175598
2,1.102408
3,1.081527
4,1.093801
5,1.180298
6,1.144196
7,1.079777
8,1.031335
9,1.059861
10,1.044311


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=15, training_loss=1.0743673006693522, metrics={'train_runtime': 33.7327, 'train_samples_per_second': 0.889, 'train_steps_per_second': 0.445, 'total_flos': 248380801920.0, 'train_loss': 1.0743673006693522, 'epoch': 3.0})